In [ ]:
import zarr
import tifffile as tiff
import pywt
import os
import sys

p = os.path.abspath('../..')
if p not in sys.path:
    sys.path.append(p)
    
    
from waveorder.util import wavelet_softThreshold
from waveorder.waveorder_reconstructor import waveorder_microscopy as setup
from recOrder.recOrder.compute.QLIPP_compute import initialize_reconstructor
import cupy as cp

In [ ]:
z_store = zarr.open('/gpfs/CompMicro/projects/A549/210203_40x_04_NA_A549/24hr_Mock.zarr', mode='r')
print(z_store.tree())

In [ ]:
## Set Reconstruction Parameters
image_dim = (2048,2448)
wavelength = 525
swing = 0.05
N_channel = 4
NA_obj = 1.2
NA_illu = 0.4
mag = 40
N_slices = 65
z_step = 0.25
pad_z = 0
pixel_size = 3.45
bg_option = 'local_fit'
n_media = 1.33

recon = initialize_reconstructor(image_dim, wavelength, swing, N_channel, NA_obj, NA_illu, mag, N_slices, z_step, pad_z, 
                             pixel_size, bg_option = bg_option, n_media = n_media, use_gpu=False, gpu_id = 0)

In [ ]:
BF = z_store['C1']['BF']

In [ ]:
BF

In [ ]:
S0_stack = BF[0]

In [ ]:
method='TV'
reg_re = 1e-4
reg_im = 1e-4
rho = 1e-5
lambda_re = 1e-3
lambda_im = 1e-3
itr = 20

In [ ]:
%%time
phase3d =  recon.Phase_recon_3D(S0_stack, method=method, reg_re = reg_re, reg_im = reg_im,\
                       rho = rho, lambda_re = lambda_re, lambda_im = lambda_im, itr = itr, verbose=True)